First we need to download sample images usnig Web Scrapping

In [1]:
import requests
from bs4 import BeautifulSoup
import os

def download_images(search_term, num_images=100):
    url = f"https://www.google.com/search?q={search_term}&tbm=isch"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    image_tags = soup.find_all("img")
    os.makedirs(search_term, exist_ok=True)

    count = 0
    for img_tag in image_tags:
        if count >= num_images:
            break
        try:
            img_url = img_tag['src']
            img_data = requests.get(img_url).content
            with open(f"{search_term}/{search_term}_{count}.jpg", "wb") as handler:
                handler.write(img_data)
                count += 1
        except Exception as e:
            print(f"Could not download {img_url}: {e}")
            
download_images("plants", num_images=100)


Could not download /images/branding/searchlogo/1x/googlelogo_desk_heirloom_color_150x55dp.gif: Invalid URL '/images/branding/searchlogo/1x/googlelogo_desk_heirloom_color_150x55dp.gif': No scheme supplied. Perhaps you meant https:///images/branding/searchlogo/1x/googlelogo_desk_heirloom_color_150x55dp.gif?


# Image processing

In [7]:
from PIL import Image
import os
import numpy as np

input_folder = 'plants'       
output_folder = 'Resized_Plants'
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".jpg"):
        img_path = os.path.join(input_folder, filename)
        img = Image.open(img_path)

        img_resized = img.resize((224, 224))

        img_normalized = np.array(img_resized) / 255.0

        img_resized.save(os.path.join(output_folder, filename))

print("Resized images saved to 'Resized_Plants' and normalized for ML processing.")



Resized images saved to 'Resized_Plants' and normalized for ML processing.


# Image Augmentation

In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Paths for saving augmented images
augmented_folder = 'Augmented_Images'
os.makedirs(augmented_folder, exist_ok=True)

datagen = ImageDataGenerator(
    rotation_range=45,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.5, 1.5],
    shear_range=0.2,
    zoom_range=[0.8, 1.2],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

for filename in os.listdir(output_folder):
    img = tf.keras.preprocessing.image.load_img(os.path.join(output_folder, filename))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = x.reshape((1,) + x.shape)

    # Generate five augmented images per original image
    i = 0
    for batch in datagen.flow(x, batch_size=1, save_to_dir=augmented_folder, save_prefix='aug', save_format='jpeg'):
        i += 1
        if i >= 5:  
            break

print("Augmented images saved to Augmented_Images.")


Augmented images saved to Augmented_Images.
